# Notes


LunarLander:

States: 8 continuous values
Actions: 4 discrete
Solved: average reward $\geq$ 200 over 100 episodes

# -

In [ ]:
# TEMP ENV SETUP
# Change to actual env when it's ready

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt

env = gym.make("LunarLander-v3")
state, _ = env.reset()
print("State shape:", state.shape)   
print("Action space:", env.action_space.n) 

State shape: (8,)
Action space: 4


In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim=8, action_dim=4, hidden_dim=128):
        super(PolicyNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )
    
    def forward(self, x):
        logits = self.network(x)
        return Categorical(logits=logits)  # returns a distribution

In [ ]:
def select_action(policy, state):
    state_tensor = torch.FloatTensor(state)
    dist = policy(state_tensor)
    action = dist.sample()                    # sample from distribution
    log_prob = dist.log_prob(action)          # save this for the loss later
    return action.item(), log_prob

In [ ]:
def compute_returns(rewards, gamma=0.99):
    returns = []
    G = 0
    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)
    
    returns = torch.FloatTensor(returns)
    
    # Normalize to reduce variance
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    return returns